# 05 · Cifras de nódulos, pacientes y cortes por regla (`scripts/contar_cortes.py`)

Este cuaderno ejecuta el script oficial que consolida las cifras de la memoria y las explora.

- **Cortes de unión**: cortes donde dibujó contorno algún radiólogo.
- **Cortes de consenso (S2)**: cortes con máscara de consenso (≥ 50 % de los lectores) de área ≥ `area_minima_corte_mm2` (2,5 mm² por defecto, valor provisional).

Requisitos: entorno TFM con `pip install -e ".[dev]"` y `setuptools<81`. La primera ejecución tarda unos 3 minutos; después usa la caché `resultados/datos/cortes_por_corte_v1.csv`.

## 1. Situar el cuaderno en la raíz del repositorio
El script usa rutas relativas a la raíz (`configs/base.yaml`, `resultados/datos`). Si el cuaderno se abrió desde `notebooks/`, subimos una carpeta.

In [1]:
import os, pathlib
# Si la carpeta actual es 'notebooks', subir a la raíz del repositorio
if pathlib.Path.cwd().name == 'notebooks':
    os.chdir('..')
print('Carpeta de trabajo:', pathlib.Path.cwd())

Carpeta de trabajo: /Users/jefferson.rodriguez/Documents/UNIR_II_Cuatrimestre/TFM/tfm-lidc-xai


## 2. Ejecutar el script con la verificación
`--verificar` compara el resultado con las cifras escritas en la memoria (Tabla 17). Si algo difiere, la celda falla y muestra las dos versiones.

In [2]:
!python -W ignore scripts/contar_cortes.py --config configs/base.yaml --verificar


Consenso ≥ 50% de los lectores; área mínima por corte = 2.5 mm²

regla  nodulos  baja  alta  pacientes  cortes_consenso  cortes_union  nodulos_sin_corte
   R1      869   448   421        530             6070          6852                  0
   R2     1180   662   518        633             7445          8416                  0
   R3     2637  2004   633        870            14173         15571                  2

✓ Coinciden con las cifras finales del TFM (R1: 869 nódulos, 530 pacientes, 6.070 cortes de consenso y 6.852 de unión).


## 3. Cargar la caché por corte
Una fila por nódulo y corte: cuántos lectores dibujaron contorno (`n_lectores`) y el área de la máscara de consenso (`area_consenso_mm2`).

In [3]:
import sys
import pandas as pd
sys.path.insert(0, 'scripts')
import contar_cortes as cc

por_corte = pd.read_csv('resultados/datos/cortes_por_corte_v1.csv')
print(len(por_corte), 'filas (incluye los 14 clústeres con más de 4 lecturas)')
por_corte.head()

15796 filas (incluye los 14 clústeres con más de 4 lecturas)


,nodule_id,k_corte,n_lectores,area_consenso_mm2
0,LIDC-IDRI-0001_s12_n01,86,3,129.528809
1,LIDC-IDRI-0001_s12_n01,87,4,328.765869
2,LIDC-IDRI-0001_s12_n01,88,4,390.563965
3,LIDC-IDRI-0001_s12_n01,89,4,456.811523
4,LIDC-IDRI-0001_s12_n01,90,4,439.508057


## 4. Sensibilidad al área mínima
Cuántos cortes de consenso quedan en cada regla según el área mínima. 2,5 mm² es la decisión provisional del equipo; la fila 0 equivale a exigir solo que la máscara no esté vacía.

In [4]:
from tfm_lidc import config
from tfm_lidc.data import nodulos as dn
from pathlib import Path

cfg = config.cargar('configs/base.yaml')
nod = dn.construir_o_cargar(cfg, Path('resultados/datos'))

filas = []
for area in (0, 2, 2.5, 3, 5, 10):
    por_nodulo = cc.resumen_por_nodulo(por_corte, area)
    x = nod.merge(por_nodulo, on='nodule_id', how='left').fillna({'cortes_union': 0, 'cortes_consenso': 0})
    fila = {'area_minima_mm2': area}
    for r in ('R1', 'R2', 'R3'):
        d = x[x[f'etiqueta_{r.lower()}'].notna()]
        fila[f'{r} consenso'] = int(d.cortes_consenso.sum())
    filas.append(fila)
pd.DataFrame(filas)

,area_minima_mm2,R1 consenso,R2 consenso,R3 consenso
0,0.0,6158,7564,14523
1,2.0,6088,7472,14270
2,2.5,6070,7445,14173
3,3.0,6053,7420,14060
4,5.0,5933,7248,13434
5,10.0,5529,6639,11553


**Lectura.** Con 2,5 mm² salen 6.070 / 7.445 / 14.173 (R1 / R2 / R3). La cifra de R3 de la memoria puede cambiar a 14.162 si el equipo adopta el cálculo con remuestreo a 0,5 mm (pendiente de incorporar al repositorio). Los cortes de unión (6.852 / 8.416 / 15.571) no dependen del umbral.